In [3]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Опционально: настройка CUDA памяти
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# ========== Конфигурация ==========
MODEL_NAME = "ai-sage/GigaChat3-10B-A1.8B-bf16"
INPUT_CSV = "../data/public/shiza.csv"
OUTPUT_DIR = "../model/"
PROBE_LAYERS = [0, 5, 10, 15, 20, 25]
NUM_ROWS = 5000               # сколько строк исходного датасета использовать (можно изменить)
CHUNK_SIZE = 5               # обрабатывать по 5 строк за раз
CHECKPOINT_PATH = OUTPUT_DIR + "checkpoint.pkl"

In [6]:
# ========== Загрузка данных ==========
df = pd.read_csv(INPUT_CSV)
df_sample = df.head(NUM_ROWS).reset_index(drop=True)
print(f"Всего строк: {len(df_sample)} → примеров: {len(df_sample)}")

# ========== Загрузка модели (8-bit) ==========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    offload_folder="offload",
)
model.eval()

Всего строк: 2000 → примеров: 2000


config.json: 0.00B [00:00, ?B/s]

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

...

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

DeepseekV3ForCausalLM LOAD REPORT from: ai-sage/GigaChat3-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.mlp.experts.down_proj               | UNEXPECTED |  | 
model.layers.26.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.26.shared_head.norm.weight             | UNEXPECTED |  | 
model.layers.26.shared_head.head.weight             | UNEXPECTED |  | 
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.gate_proj.weight | UNEXPECTED |  | 
model.layers.26.mlp.gate.e_score_correction_bias    | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_proj_with_mqa.weight | UNEXPECTED |  | 
model.layers.26.embed_tokens.weight                 | UNEXPECTED |  | 
model.layers.26.eh_proj.weight                      | UNEXPECTED |  | 
model.layers.26.input_layernorm.weight              | UNEXPECTED |  | 
mode

generation_config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

DeepseekV3ForCausalLM(
  (model): DeepseekV3Model(
    (embed_tokens): Embedding(128256, 1536)
    (layers): ModuleList(
      (0): DeepseekV3DecoderLayer(
        (self_attn): DeepseekV3Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=6144, bias=False)
          (kv_a_proj_with_mqa): Linear8bitLt(in_features=1536, out_features=576, bias=False)
          (kv_a_layernorm): DeepseekV3RMSNorm((512,), eps=1e-06)
          (kv_b_proj): Linear8bitLt(in_features=512, out_features=10240, bias=False)
          (o_proj): Linear8bitLt(in_features=6144, out_features=1536, bias=False)
        )
        (mlp): DeepseekV3MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): DeepseekV3RMSNorm((1536,), e

In [7]:
# ========== FeatureAccumulator (как в baseline) ==========
class FeatureAccumulator:
    def __init__(self, model, probe_layers):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks = []
        self._hidden = {}
    def attach(self):
        self._hidden.clear()
        layers = self.model.model.layers
        for idx in self.probe_layers:
            if idx >= len(layers): continue
            name = f"layer_{idx}"
            def make_hook(n):
                def hook(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return hook
            self._hooks.append(layers[idx].register_forward_hook(make_hook(name)))
    def detach(self):
        for h in self._hooks: h.remove()
        self._hooks.clear()
    def __enter__(self):
        self.attach()
        return self
    def __exit__(self, *args):
        self.detach()
    def compute_features(self, logits, input_ids, ans_start):
        seq_len = input_ids.shape[1]
        ans_len = seq_len - ans_start

        # --- Uncertainty features ---
        answer_logits = logits[0, ans_start-1:seq_len-1, :].float()
        answer_ids = input_ids[0, ans_start:seq_len]
        answer_ids = answer_ids.to(answer_logits.device)

        log_probs = F.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)

        probs = F.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)

        uncertainty = np.array([
            token_lp.mean().item(), token_lp.min().item(), token_lp.max().item(),
            token_lp.std().item() if ans_len > 1 else 0.0,
            entropy.mean().item(), entropy.max().item(),
            entropy.std().item() if ans_len > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(), float(ans_len), token_lp[0].item(),
            top1.mean().item(), top5.mean().item()
        ], dtype=np.float32)

        # --- Internal features & probe vector ---
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vec = last_hs[ans_start-1].cpu().float().numpy()

        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[ans_start-1].norm().item())
            int_scalars.append(hs[ans_start:seq_len].norm(dim=-1).mean().item())

            ans_hs = hs[ans_start-1:seq_len-1]  # [L, D]
            L = ans_hs.shape[0]
            chunk_size = 512
            ll_e_sum = 0.0
            num_chunks = 0
            for start in range(0, L, chunk_size):
                end = min(start + chunk_size, L)
                chunk = ans_hs[start:end].unsqueeze(0)  # [1, chunk_len, D]
                with torch.no_grad():
                    norm_chunk = self.model.model.norm(chunk)
                    ll_chunk = self.model.lm_head(norm_chunk).float().cpu()
                ll_p_chunk = torch.softmax(ll_chunk[0], dim=-1)
                ll_e_chunk = -(ll_p_chunk * torch.log(ll_p_chunk + 1e-10)).sum(dim=-1)
                ll_e_sum += ll_e_chunk.sum().item()
                num_chunks += (end - start)
                del chunk, norm_chunk, ll_chunk, ll_p_chunk, ll_e_chunk
                torch.cuda.empty_cache()
            int_scalars.append(ll_e_sum / num_chunks if num_chunks > 0 else 0.0)

        first_e, last_e = int_scalars[2], int_scalars[-1]
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))

        internal = np.array(int_scalars, dtype=np.float32)
        self._hidden.clear()
        return {"uncertainty": uncertainty, "internal_scalars": internal, "probe_vec": probe_vec}

In [8]:
# ========== Препроцессинг (адаптирован под новый формат) ==========
def preprocess(tokenizer, prompt, answer):
    # Применяем chat template: сначала user, потом assistant
    prompt_enc = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True
    )
    ans_start = len(prompt_enc["input_ids"])

    full_enc = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
        tokenize=True
    )
    tok_ids = torch.tensor([full_enc["input_ids"]], dtype=torch.long)
    return tok_ids, ans_start


In [9]:
# ========== Восстановление чекпоинта ==========
start_idx = 0
all_unc = []; all_int = []; all_probe = []; all_labels = []
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        start_idx, all_unc, all_int, all_probe, all_labels = pickle.load(f)
    print(f"Восстановлен чекпоинт: обработано {start_idx} строк")
else:
    print("Новый запуск")


Новый запуск


In [10]:
# ========== Обработка чанками ==========
accumulator = FeatureAccumulator(model, PROBE_LAYERS)

for chunk_start in range(start_idx, len(df_sample), CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, len(df_sample))
    print(f"\nОбработка строк {chunk_start}–{chunk_end-1}...")

    for idx in tqdm(range(chunk_start, chunk_end), desc="Внутри чанка"):
        row = df_sample.iloc[idx]
        prompt = row['prompt']               # промпт уже готов
        answer = row['model_answer']         # ответ модели
        label = 1 if row['is_hallucination'] else 0

        # Один проход на каждый пример
        tok, start = preprocess(tokenizer, prompt, answer)
        with accumulator, torch.no_grad():
            out = model(tok.to(model.device))
        f = accumulator.compute_features(out.logits, tok, start)

        all_unc.append(f["uncertainty"])
        all_int.append(f["internal_scalars"])
        all_probe.append(f["probe_vec"])
        all_labels.append(label)

        del out, tok, f
        accumulator._hidden.clear()
        torch.cuda.empty_cache()

    # Сохраняем чекпоинт после чанка
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump((chunk_end, all_unc, all_int, all_probe, all_labels), f)
    print(f"Чекпоинт сохранён до строки {chunk_end}")
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Собрано примеров: {len(all_labels)}")



Обработка строк 0–4...



Внутри чанка: 100%|██████████| 5/5 [00:13<00:00,  2.71s/it]


Чекпоинт сохранён до строки 5

Обработка строк 5–9...



Внутри чанка: 100%|██████████| 5/5 [00:08<00:00,  1.75s/it]


Чекпоинт сохранён до строки 10

Обработка строк 10–14...


...

Чекпоинт сохранён до строки 1990

Обработка строк 1990–1994...



Внутри чанка: 100%|██████████| 5/5 [00:09<00:00,  1.90s/it]


Чекпоинт сохранён до строки 1995

Обработка строк 1995–1999...



Внутри чанка: 100%|██████████| 5/5 [00:10<00:00,  2.12s/it]


Чекпоинт сохранён до строки 2000

✅ Собрано примеров: 2000


In [11]:
# ========== Преобразование в массивы ==========
print("Преобразование в массивы...")
unc = np.stack(all_unc)           # (n_samples, 12)
ints = np.stack(all_int)          # (n_samples, N_int)
probe = np.stack(all_probe)       # (n_samples, hidden_dim)
y = np.array(all_labels)          # (n_samples,)

# ========== Стратифицированное разделение на TRAIN/TEST ==========
X_raw = np.hstack([probe, ints, unc])   # объединяем все признаки
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# ========== PCA только на TRAIN ==========
print("PCA (64 компоненты) на train...")
pca = PCA(n_components=64, random_state=42)
probe_pca_train = pca.fit_transform(X_train[:, :probe.shape[1]])
probe_pca_test = pca.transform(X_test[:, :probe.shape[1]])

X_train_pca = np.hstack([probe_pca_train, X_train[:, probe.shape[1]:]])
X_test_pca = np.hstack([probe_pca_test, X_test[:, probe.shape[1]:]])

# ========== StandardScaler только на TRAIN ==========
print("Масштабирование на train...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled = scaler.transform(X_test_pca)

# ========== Сохранение ==========
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.savez_compressed(OUTPUT_DIR + "features_train.npz", X=X_train_scaled, y=y_train)
np.savez_compressed(OUTPUT_DIR + "features_test.npz", X=X_test_scaled, y=y_test)

with open(OUTPUT_DIR + "pca.pkl", "wb") as f:
    pickle.dump(pca, f)
with open(OUTPUT_DIR + "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"\n🎉 Готово! Файлы сохранены в {OUTPUT_DIR}")
print(f"Train X: {X_train_scaled.shape}, y: {np.bincount(y_train)}")
print(f"Test X: {X_test_scaled.shape}, y: {np.bincount(y_test)}")

# Удаляем чекпоинт
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

Преобразование в массивы...
Train size: (1600, 1568), Test size: (400, 1568)
PCA (64 компоненты) на train...
Масштабирование на train...

🎉 Готово! Файлы сохранены в /kaggle/working/
Train X: (1600, 96), y: [800 800]
Test X: (400, 96), y: [200 200]
